# Исследовательский обзор покрытия меток

Цель:
- Проверить покрытие `spec_class`, `evolution_stage` и `spec_subclass` в локальных обучающих источниках.
- Оценить, достаточно ли текущих таблиц базы для будущей волны subclass.
- Понять, где достаточно локального источника, а где действительно придется идти за дополнительной выборкой в Gaia.

In [1]:
# Настройка: корень репозитория, sys.path и базовые визуальные параметры.
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    # Ищем корень репозитория по наличию src и analysis.
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").exists() and (candidate / "analysis").exists():
            return candidate
    raise RuntimeError("Не удалось определить корень репозитория из текущей рабочей директории.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style="whitegrid", context="notebook")



In [2]:
# Импортируем V2-модули только после добавления src в sys.path.
from exohost.datasets.load_host_training_dataset import load_host_training_dataset
from exohost.datasets.load_router_training_dataset import load_router_training_dataset
from exohost.db.engine import make_read_only_engine
from exohost.features.training_frame import (
    prepare_host_training_frame,
    prepare_router_training_frame,
)



## Что делаем

- Загружаем оба основных обучающих источника: `router` и `host`.
- Смотрим покрытие coarse-меток и долю непустого `spec_subclass`.
- Строим сводные таблицы по классам, стадиям и подклассам.
- Формулируем инженерный вывод: хватает ли локальной базы для следующей волны или нужен дополнительный запрос в Gaia.

In [3]:
# Конфигурация notebook.
DOTENV_PATH = ".env"
ROUTER_LIMIT: int | None = None
HOST_LIMIT: int | None = None
TOP_SUBCLASS_ROWS = 30


def build_subclass_coverage_frame(df: pd.DataFrame, *, source_name: str) -> pd.DataFrame:
    # Собираем русифицированную таблицу покрытия по классам и подклассам.
    result = (
        df.assign(spec_subclass_notna=df["spec_subclass"].notna() if "spec_subclass" in df.columns else False)
        .groupby("spec_class", dropna=False)
        .agg(
            n_rows=("source_id", "count"),
            n_subclass_labeled=("spec_subclass_notna", "sum"),
            n_distinct_subclass=("spec_subclass", lambda s: int(s.dropna().astype(str).nunique())),
        )
        .reset_index()
        .rename(
            columns={
                "spec_class": "Спектральный класс",
                "n_rows": "Число строк",
                "n_subclass_labeled": "Строк с размеченным подклассом",
                "n_distinct_subclass": "Число разных подклассов",
            }
        )
    )
    result["Доля строк с размеченным подклассом"] = (
        result["Строк с размеченным подклассом"] / result["Число строк"]
    )
    result.insert(0, "Источник", source_name)
    return result.sort_values("Спектральный класс", ignore_index=True)


def build_subclass_frequency_frame(df: pd.DataFrame, *, source_name: str) -> pd.DataFrame:
    # Собираем частоты подклассов для уже размеченных строк.
    if "spec_subclass" not in df.columns:
        return pd.DataFrame(columns=["Источник", "Спектральный класс", "Подкласс", "Число строк"])

    labeled_df = df.loc[df["spec_subclass"].notna()].copy()
    if labeled_df.empty:
        return pd.DataFrame(columns=["Источник", "Спектральный класс", "Подкласс", "Число строк"])

    frequency_df = (
        labeled_df.groupby(["spec_class", "spec_subclass"], dropna=False)["source_id"]
        .count()
        .reset_index(name="Число строк")
        .sort_values(["spec_class", "Число строк", "spec_subclass"], ascending=[True, False, True], ignore_index=True)
        .rename(columns={"spec_class": "Спектральный класс", "spec_subclass": "Подкласс"})
    )
    frequency_df.insert(0, "Источник", source_name)
    return frequency_df


In [4]:
# Загружаем оба источника и приводим их к каноническому виду.
engine = make_read_only_engine(dotenv_path=DOTENV_PATH)
try:
    raw_router_df = load_router_training_dataset(engine, limit=ROUTER_LIMIT)
    raw_host_df = load_host_training_dataset(engine, limit=HOST_LIMIT)
finally:
    engine.dispose()

router_df = prepare_router_training_frame(raw_router_df)
host_df = prepare_host_training_frame(raw_host_df)

source_summary_df = pd.DataFrame(
    [
        {"Источник": "router", "Число строк": int(router_df.shape[0]), "Число столбцов": int(router_df.shape[1])},
        {"Источник": "host", "Число строк": int(host_df.shape[0]), "Число столбцов": int(host_df.shape[1])},
    ]
)

display(source_summary_df)


,Источник,Число строк,Число столбцов
0,router,39413,16
1,host,3729,24


In [5]:
# Покрытие coarse-классов и разметки подклассов.
router_coverage_df = build_subclass_coverage_frame(router_df, source_name="router")
host_coverage_df = build_subclass_coverage_frame(host_df, source_name="host")
coverage_df = pd.concat([router_coverage_df, host_coverage_df], ignore_index=True)

display(coverage_df)


if float(coverage_df["Доля строк с размеченным подклассом"].max()) == 0.0:
    print(
        "Текущие локальные источники router и host не дают пригодного покрытия по подклассам. "
        "Это ограничение источника данных, а не проблема отображения notebook."
    )


,Источник,Спектральный класс,Число строк,Строк с размеченным подклассом,Число разных подклассов,Доля строк с размеченным подклассом
0,router,A,5097,0,0,0.0
1,router,B,4975,0,0,0.0
2,router,F,6000,0,0,0.0
3,router,G,6000,0,0,0.0
4,router,K,6000,0,0,0.0
5,router,M,6000,0,0,0.0
6,router,O,5341,0,0,0.0
7,host,A,20,0,0,0.0
8,host,B,6,0,0,0.0
9,host,F,471,0,0,0.0


Текущие локальные источники router и host не дают пригодного покрытия по подклассам. Это ограничение источника данных, а не проблема отображения notebook.


In [6]:
# Покрытие стадий и сводные таблицы по coarse-классам.
router_stage_df = pd.crosstab(router_df["spec_class"], router_df["evolution_stage"]).rename_axis(
    index="Спектральный класс", columns="Стадия эволюции"
)
host_stage_df = pd.crosstab(host_df["spec_class"], host_df["evolution_stage"]).rename_axis(
    index="Спектральный класс", columns="Стадия эволюции"
)

display(router_stage_df)
display(host_stage_df)


Стадия эволюции,dwarf,evolved
Спектральный класс,,
A,2097,3000
B,1975,3000
F,3000,3000
G,3000,3000
K,3000,3000
M,3000,3000
O,2341,3000


Стадия эволюции,dwarf,evolved
Спектральный класс,,
A,15,5
B,4,2
F,411,60
G,1830,115
K,925,133
M,221,8


In [7]:
# Частоты конкретных подклассов, если они реально присутствуют.
router_subclass_df = build_subclass_frequency_frame(router_df, source_name="router")
host_subclass_df = build_subclass_frequency_frame(host_df, source_name="host")

if router_subclass_df.empty:
    print("В источнике router сейчас нет пригодных меток подклассов.")
else:
    display(router_subclass_df.head(TOP_SUBCLASS_ROWS))

if host_subclass_df.empty:
    print("В источнике host сейчас нет пригодных меток подклассов.")
else:
    display(host_subclass_df.head(TOP_SUBCLASS_ROWS))


В источнике router сейчас нет пригодных меток подклассов.
В источнике host сейчас нет пригодных меток подклассов.


In [8]:
# Визуальная проверка: доля заполненного подкласса по coarse-классам.
max_subclass_share = float(coverage_df['Доля строк с размеченным подклассом'].max())
if max_subclass_share == 0.0:
    print(
        'Последний график не строим: в текущих локальных источниках нет пригодного '
        'покрытия по спектральным подклассам.'
    )
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(
        data=coverage_df,
        x='Спектральный класс',
        y='Доля строк с размеченным подклассом',
        hue='Источник',
        ax=ax,
    )
    ax.set_title('Покрытие подклассов по крупным спектральным классам')
    ax.set_xlabel('Спектральный класс')
    ax.set_ylabel('Доля строк с размеченным подклассом')
    plt.tight_layout()


Последний график не строим: в текущих локальных источниках нет пригодного покрытия по спектральным подклассам.


## Что смотреть в выводе

- Достаточно ли у `router` и `host` покрытия по `spec_subclass`?
- Есть ли смысл продолжать только с локальной базой, или уже нужен дополнительный источник?
- Главный вывод здесь не модельный, а договорный: хватает ли текущей базы для следующего шага?